# UMA on Kaggle: broadened SAC set + finishing the S/N set

This notebook runs both remaining UMA jobs in one go, using Kaggle's free GPUs:

1. **Broadened set**: 280 new catalysts (N/C on the divacancy, single-vacancy sites), in tiers core → distal → full.
2. **S/N set**: the remaining full-pathway intermediates for the 120 S/N catalysts not finished on Colab.
   Your earlier Colab results are already inside the bundle.

**Why Kaggle.** A *GPU T4 x2* session has two GPUs, so the notebook runs two workers in parallel: one catalyst per GPU,
which roughly halves the time. A committed run continues in the background with the browser closed. It stops cleanly
before the time budget runs out, saves everything to the Output, and the next version resumes from that Output.

## One-time setup
1. Kaggle account with a **verified phone number** (needed for GPU and internet).
2. **Datasets → New Dataset**: upload `nrr_kaggle_bundle.zip`, name it e.g. `nrr-sac-bundle`, keep it private, create.
3. **New Notebook → File → Import Notebook**: upload this file.
4. Right panel: **Add Input** → your dataset `nrr-sac-bundle`.
5. Right panel, **Session options**: Accelerator **GPU T4 x2**, Internet **On**.
6. **Add-ons → Secrets → Add secret**: label `HF_TOKEN`, value = your Hugging Face token. Tick it for this notebook.

## Each run
7. **Save Version → Save & Run All (Commit) → Save**. You can close the browser.
8. When it finishes, open the version → **Output**. Download `results_for_claude.zip` and check `progress.txt`.
9. If `progress.txt` says something is unfinished: **Add Input** → this notebook's own previous version (its output),
   then Save & Run All again. It merges the old output and continues. Repeat until everything is complete.

## 1. Settings

In [ ]:
import os, sys, json, time, glob, shutil, zipfile, subprocess
T_START = time.time()
MODEL = 'uma'              # 'uma' | 'mace' (no-login stand-in, for testing only)
UMA_MODEL, TASK = 'uma-s-1p1', 'omat'
TIME_BUDGET_H = 8.5        # stop starting new catalysts after this (packing needs ~10 min); 11 if your session limit is 12 h
BROADENED_UPTO = 'full'    # 'core' | 'distal' | 'full'
FINISH_SN = True           # finish the full pathway of the S/N set
N_WORKERS = 'auto'         # 'auto' = one per GPU
TEST_MODE = False          # True = 2 catalysts per job (check the notebook quickly)
IN_KAGGLE = os.path.isdir('/kaggle/input')
INPUT_ROOT = os.environ.get('NRR_INPUT_ROOT', '/kaggle/input')
WORK = os.environ.get('NRR_WORK', '/kaggle/working')
RUN = f'{WORK}/run'
os.makedirs(RUN, exist_ok=True)
print('Kaggle:', IN_KAGGLE, '| input:', INPUT_ROOT, '| work:', WORK)

## 2. Install UMA and read the Hugging Face token

In [ ]:
if IN_KAGGLE:
    pkgs = ['fairchem-core'] if MODEL == 'uma' else ['mace-torch']
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
if MODEL == 'uma' and IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN read from Kaggle secrets')
    except Exception as e:
        print('Could not read the HF_TOKEN secret - add it under Add-ons > Secrets:', e)
gpus = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines() if shutil.which('nvidia-smi') else []
print('GPUs:', gpus or 'none')

## 3. Find the bundle and restore earlier progress

In [ ]:
def find_bundle():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if 'kaggle_bundle.marker' in files:
            return root
    for z in glob.glob(f'{INPUT_ROOT}/**/*.zip', recursive=True):
        with zipfile.ZipFile(z) as f:
            if any(n.endswith('kaggle_bundle.marker') for n in f.namelist()):
                dest = '/tmp/nrr_bundle'
                if not os.path.isdir(dest):
                    f.extractall(dest)
                for root, dirs, files in os.walk(dest):
                    if 'kaggle_bundle.marker' in files:
                        return root
    raise FileNotFoundError('Bundle not found: add the dataset with nrr_kaggle_bundle.zip as an input')

BUNDLE = find_bundle()
sys.path.insert(0, BUNDLE)
import pandas as pd
import nrr_mlip as nm
print('bundle:', BUNDLE)

def read_rows(paths):
    rows = {}
    for p in paths:
        if os.path.exists(p):
            for line in open(p):
                if line.strip():
                    r = json.loads(line); rows[(r['system'], r['kind'])] = r      # later files win
    return list(rows.values())

def restore(job, seed=None):
    # every earlier result for this job: the bundle seed, a previous version's output, this session
    res = ([seed] if seed else [])
    res += glob.glob(f'{INPUT_ROOT}/**/out/{job}/results.jsonl', recursive=True)
    res += glob.glob(f'{INPUT_ROOT}/**/run/{job}_w*/results.jsonl', recursive=True)
    res += glob.glob(f'{WORK}/out/{job}/results.jsonl') + glob.glob(f'{RUN}/{job}_w*/results.jsonl')
    rows = read_rows(res)
    zips = glob.glob(f'{INPUT_ROOT}/**/out/{job}/relaxed.zip', recursive=True) + glob.glob(f'{WORK}/out/{job}/relaxed.zip')
    dirs = glob.glob(f'{INPUT_ROOT}/**/run/{job}_w*/relaxed', recursive=True) + glob.glob(f'{RUN}/{job}_w*/relaxed')
    stage = f'/tmp/restore_{job}'
    shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
    for z in zips:
        zipfile.ZipFile(z).extractall(stage)
    for d in dirs:
        shutil.copytree(d, stage, dirs_exist_ok=True)
    return rows, stage

def seed_workers(job, rows, stage, nw):
    for k in range(nw):
        d = f'{RUN}/{job}_w{k}'
        os.makedirs(d, exist_ok=True)
        with open(f'{d}/results.jsonl', 'w') as f:
            for r in rows:
                f.write(json.dumps(r) + '\n')
        shutil.copytree(stage, f'{d}/relaxed', dirs_exist_ok=True)

NW = (max(len(gpus), 1) if N_WORKERS == 'auto' else int(N_WORKERS))
PKG_B, PKG_S = f'{BUNDLE}/SAC_broadened_package', f'{BUNDLE}/SAC_UMA_package'
rows_b, stage_b = restore('broadened')
rows_s, stage_s = restore('sn', seed=f'{BUNDLE}/previous/results_omat_SN.jsonl')
for j in ('broadened', 'sn'):
    for d in glob.glob(f'{RUN}/{j}_w*'):
        shutil.rmtree(d)
seed_workers('broadened', rows_b, stage_b, NW)
seed_workers('sn', rows_s, stage_s, NW)
print(f'workers: {NW} | restored rows: broadened {len(rows_b)}, S/N {len(rows_s)}')

## 4. Plan the jobs

In [ ]:
CORE = ['H', 'N2_end', 'NNH_end', 'NH2']
DISTAL = CORE + ['NNH2', 'N', 'NH', 'NH3']
FULL = DISTAL + ['N2_side', 'NNH_side', 'NHNH', 'NHNH2', 'NH2NH2']
man_b = pd.read_csv(f'{PKG_B}/manifest.csv'); man_s = pd.read_csv(f'{PKG_S}/manifest.csv')
SACS_B = sorted(man_b[man_b.kind == 'bare'].system)
SACS_S = sorted(man_s[man_s.kind == 'bare'].system)
refs = man_b[man_b.tier == '0_reference']
if TEST_MODE:
    SACS_B = ['Fe@N2C2-PEN', 'Fe@SV-N3']
    have = {r['system'] for r in rows_s if r['kind'] == 'NNH2'}
    SACS_S = [s for s in SACS_S if s not in have][:2]
    refs = refs[(refs.kind == 'gas') | refs.system.isin(['support_N2C2-PEN', 'support_SV-N3', 'Fe'])]
order = ['core', 'distal', 'full']
jobs = [{'type': 'refs', 'name': 'broadened', 'label': 'broadened references',
         'items': [{'kind': r.kind, 'system': r.system, 'path': f'{PKG_B}/{r.file}'} for r in refs.itertuples()]}]
tiers = {'core': CORE, 'distal': DISTAL, 'full': FULL}
for t in order[:order.index(BROADENED_UPTO) + 1]:
    if t == 'full' and FINISH_SN:
        jobs.append({'type': 'sacs', 'name': 'sn', 'label': 'S/N full pathway', 'sacs': SACS_S,
                     'ads': FULL, 'struct_dir': f'{PKG_S}/structures', 'rebuild': False})
    jobs.append({'type': 'sacs', 'name': 'broadened', 'label': f'broadened {t}', 'sacs': SACS_B,
                 'ads': tiers[t], 'struct_dir': f'{PKG_B}/structures', 'rebuild': True})
if FINISH_SN and BROADENED_UPTO != 'full':
    jobs.append({'type': 'sacs', 'name': 'sn', 'label': 'S/N full pathway', 'sacs': SACS_S,
                 'ads': FULL, 'struct_dir': f'{PKG_S}/structures', 'rebuild': False})
deadline = T_START + TIME_BUDGET_H * 3600 - (0 if TEST_MODE else 600)
cfg = dict(bundle=BUNDLE, run_dir=RUN, model=MODEL, uma_model=UMA_MODEL, task=TASK, deadline=deadline, jobs=jobs)
json.dump(cfg, open(f'{RUN}/config.json', 'w'))
done_b = {(r['system'], r['kind']) for r in rows_b}; done_s = {(r['system'], r['kind']) for r in rows_s}
for j in jobs:
    if j['type'] == 'sacs':
        done = done_b if j['name'] == 'broadened' else done_s
        left = sum((s, a) not in done for s in j['sacs'] for a in ['bare'] + j['ads'])
        print(f"{j['label']:22s} {len(j['sacs'])} catalysts, {left} structures still to do")
print(f'time budget: {TIME_BUDGET_H} h; rough need at 6 s per structure on {NW} GPU(s): see the counts above x 6 s / {NW}')

## 5. Run the workers (one per GPU)

In [ ]:
worker = f'{BUNDLE}/kaggle_worker.py'
procs, logs = [], [f'{RUN}/worker{k}.log' for k in range(NW)]
for k in range(NW):
    env = dict(os.environ)
    if gpus:
        env['CUDA_VISIBLE_DEVICES'] = str(k)
    out = open(f'{RUN}/worker{k}.out', 'w')
    procs.append(subprocess.Popen([sys.executable, worker, f'{RUN}/config.json', str(k), str(NW)],
                                  env=env, stdout=out, stderr=subprocess.STDOUT))
    if k == 0:                    # let worker 0 download the model first
        while procs[0].poll() is None and not (os.path.exists(logs[0]) and 'model loaded' in open(logs[0]).read()):
            time.sleep(5)
seen = [0] * NW
last = 0
while any(p.poll() is None for p in procs):
    time.sleep(10 if TEST_MODE else 60)
    if time.time() - last > (20 if TEST_MODE else 600):
        last = time.time()
        for k in range(NW):
            if os.path.exists(logs[k]):
                lines = open(logs[k]).read().splitlines()
                for l in lines[seen[k]:]:
                    print(f'[worker {k}] {l}')
                seen[k] = len(lines)
for k in range(NW):
    lines = open(logs[k]).read().splitlines() if os.path.exists(logs[k]) else []
    for l in lines[seen[k]:]:
        print(f'[worker {k}] {l}')
    if procs[k].returncode != 0:
        print(f'worker {k} exited with code {procs[k].returncode}; last output:')
        print(open(f'{RUN}/worker{k}.out').read()[-3000:])
print(f'elapsed {(time.time() - T_START) / 3600:.2f} h')

## 6. Pack the output, analyse, and report progress

In [ ]:
def pack(job):
    files = sorted(glob.glob(f'{RUN}/{job}_w*/results.jsonl'))
    rows = read_rows(files)
    d = f'{WORK}/out/{job}'
    os.makedirs(d, exist_ok=True)
    with open(f'{d}/results.jsonl', 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
    tmp = f'/tmp/{job}_relaxed.zip'
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
        seen = set()
        for rd in glob.glob(f'{RUN}/{job}_w*/relaxed'):
            for root, _, fs in os.walk(rd):
                for fn in fs:
                    p = os.path.join(root, fn); arc = os.path.relpath(p, rd)
                    if arc not in seen:
                        z.write(p, arc); seen.add(arc)
    shutil.move(tmp, f'{d}/relaxed.zip')
    for rd in glob.glob(f'{RUN}/{job}_w*'):
        shutil.rmtree(rd)
    return rows

rows_b, rows_s = pack('broadened'), pack('sn')
lines = []
done = {'broadened': {(r['system'], r['kind']) for r in rows_b}, 'sn': {(r['system'], r['kind']) for r in rows_s}}
complete = True
for j in jobs:
    if j['type'] == 'sacs':
        need = [(s, a) for s in j['sacs'] for a in ['bare'] + j['ads']]
        left = sum(x not in done[j['name']] for x in need)
        complete &= left == 0
        lines.append(f"{j['label']:22s} {'COMPLETE' if left == 0 else f'{left} structures left'}")
lines.append('ALL DONE - send results_for_claude.zip' if complete else
             'NOT FINISHED - add this version\'s output as an input and Save & Run All again')
open(f'{WORK}/progress.txt', 'w').write('\n'.join(lines) + '\n')
print('\n'.join(lines))

for job, name in (('broadened', 'broadened'), ('sn', 'SN')):
    try:
        ads_df, sac_df = nm.analyse(f'{WORK}/out/{job}/results.jsonl', refs='computed')
        sac_df.to_csv(f'{WORK}/{name}_SAC_results_{TASK}.csv', index=False)
        ads_df.to_csv(f'{WORK}/{name}_structure_energies_{TASK}.csv', index=False)
        print(f'{name}: {len(sac_df)} catalysts analysed')
    except Exception as e:
        print(f'{name}: analysis skipped ({e})')

with zipfile.ZipFile(f'{WORK}/results_for_claude.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{WORK}/out/broadened/results.jsonl', f'{WORK}/out/sn/results.jsonl', f'{WORK}/progress.txt'] + glob.glob(f'{WORK}/*.csv'):
        if os.path.exists(p):
            z.write(p, os.path.relpath(p, WORK))
shutil.rmtree(RUN, ignore_errors=True)
print('Output ready:', sorted(os.listdir(WORK)))